# BA Thesis – Phase 3: Full Bono Inference (Fine-Tuned, Kaggle GPU)

**Ziel:** Flächige Vorhersage auf `Bono_Merged_2025.tif` mit `prithvi-v2-300-finetuned.ckpt`.

**Outputs:** `prediction_prob.tif`, `prediction_binary.tif` → lokal nach `data/inference_bono_full_ft/` kopieren.

---

## Kaggle-Setup

1. **Dataset A – Mosaik:** `Bono_Merged_2025.tif` (~24 GB) als Kaggle Dataset hochladen (z. B. `bono-merged-2025`).
2. **Dataset B – Checkpoint:** `prithvi-v2-300-finetuned.ckpt` (z. B. `prithvi-bono-finetuned-ckpt`).
3. Dieses Notebook hochladen → **Add data** → beide Datasets.
4. Settings → Accelerator → **GPU P100/T4**, Persistence optional.
5. Nach dem Lauf: Output-Zip herunterladen → `data/inference_bono_full_ft/`.


## Zelle 1: Pakete


In [ ]:
import subprocess, sys

_np_ver = subprocess.check_output(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)']
).decode().strip()
print(f'Kaggle numpy (pin): {_np_ver}')

# Terratorch zieht typischerweise torch 2.5.x → torchvision muss dazu passen
!pip install -q terratorch==0.99.7 "torchgeo>=0.6.0,<0.7.0" numpy=={_np_ver} rasterio
!pip install -q --force-reinstall "torchvision==0.20.0" "torch==2.5.0"

import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('CUDA:', torch.cuda.is_available())
print('Install done – danach Restart Session, dann Zelle 2→3')



## Zelle 2: Pfade & Config


In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import rasterio
from rasterio.windows import Window

INPUT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working/inference_bono_full_ft')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('=== /kaggle/input Inhalt ===')
if not INPUT.exists():
    raise FileNotFoundError('/kaggle/input fehlt – Datasets im Notebook unter Add data hinzufügen.')
for p in sorted(INPUT.rglob('*')):
    if p.is_file():
        mb = p.stat().st_size / 1e6
        print(f'  {p}  ({mb:.1f} MB)')

# Rekursiv alle TIFFs / CKPTs finden (Kaggle-Dataset-Slug ≠ Dateiname)
tifs = sorted(INPUT.rglob('*.tif')) + sorted(INPUT.rglob('*.tiff'))
ckpts = sorted(INPUT.rglob('*.ckpt'))
print(f'\nGefundene TIFs ({len(tifs)}):')
for t in tifs:
    print(f'  {t}  ({t.stat().st_size/1e9:.2f} GB)')
print(f'Gefundene CKPTs ({len(ckpts)}):')
for c in ckpts:
    print(f'  {c}  ({c.stat().st_size/1e6:.1f} MB)')

def pick_mosaic(tifs):
    # Prefer Bono/Merged in name; else largest GeoTIFF (full mosaic ~20+ GB)
    prefer = [t for t in tifs if 'bono' in t.name.lower() or 'merged' in t.name.lower()]
    pool = prefer or tifs
    if not pool:
        raise FileNotFoundError(
            'Kein .tif unter /kaggle/input. '
            'Dataset mit Bono_Merged_2025.tif über Add data anbinden.'
        )
    return max(pool, key=lambda p: p.stat().st_size)

def pick_ckpt(ckpts):
    prefer = [c for c in ckpts if 'finetuned' in c.name.lower() or 'bono' in c.name.lower()]
    pool = prefer or ckpts
    if not pool:
        raise FileNotFoundError(
            'Kein .ckpt unter /kaggle/input. '
            'Dataset mit prithvi-v2-300-finetuned.ckpt über Add data anbinden.'
        )
    return prefer[0] if prefer else pool[0]

MOSAIC_PATH = pick_mosaic(tifs)
CHECKPOINT_PATH = pick_ckpt(ckpts)

LIMIT_PATCHES = int(os.environ.get('LIMIT_PATCHES', '0'))  # 0 = voll
MINING_THRESH = 0.5
PATCH_SIZE = 128
BAND_INDICES = list(range(1, 7))
MEANS = np.array([1473.81, 1703.35, 1696.68, 3832.40, 3156.11, 2226.07], dtype=np.float32)
STDS  = np.array([ 223.44,  285.54,  413.82,  389.61,  451.50,  468.27], dtype=np.float32)

print('\nMOSAIC:', MOSAIC_PATH)
print('CKPT:  ', CHECKPOINT_PATH)
print('LIMIT: ', LIMIT_PATCHES)
print('CUDA:  ', torch.cuda.is_available())


## Zelle 3: Modell laden


In [ ]:
from terratorch.tasks import SemanticSegmentationTask

bands = ['BLUE', 'GREEN', 'RED', 'VNIR_5', 'SWIR_1', 'SWIR_2']
model_args = {
    'backbone': 'prithvi_eo_v2_300',
    'bands': bands,
    'in_channels': 6,
    'num_classes': 2,
    'pretrained': False,
    'decoder': 'UperNetDecoder',
    'rescale': True,
    'backbone_num_frames': 1,
    'head_dropout': 0.1,
    'decoder_scale_modules': True,
}
task = SemanticSegmentationTask.load_from_checkpoint(
    str(CHECKPOINT_PATH),
    model_args=model_args,
    model_factory='PrithviModelFactory',
    loss='ce', lr=1e-3, ignore_index=-1,
    optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
    freeze_backbone=True,
    class_names=['Non_mining', 'Mining'],
    strict=False,
)
task.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
task = task.to(device)
print('Model on', device)


## Zelle 4: Voll-Inferenz


In [ ]:
def normalize(patch):
    patch = np.nan_to_num(patch.astype(np.float32), nan=0.0)
    return (patch - MEANS[:, None, None]) / STDS[:, None, None]

with rasterio.open(MOSAIC_PATH) as src:
    width, height = src.width, src.height
    out_h = (height // PATCH_SIZE) * PATCH_SIZE
    out_w = (width // PATCH_SIZE) * PATCH_SIZE
    n_rows = out_h // PATCH_SIZE
    n_cols = out_w // PATCH_SIZE
    n_full = n_rows * n_cols
    total = n_full if LIMIT_PATCHES <= 0 else min(LIMIT_PATCHES, n_full)
    print(f'Mosaic {width}x{height} → {n_full} patches, processing {total}')

    profile_prob = dict(
        driver='GTiff', width=out_w, height=out_h, count=1, dtype='float32',
        crs=src.crs, transform=src.transform, compress='lzw', tiled=True,
        blockxsize=128, blockysize=128, BIGTIFF='YES',
    )
    profile_binary = {**profile_prob, 'dtype': 'uint8'}
    prob_path = OUT_DIR / 'prediction_prob.tif'
    binary_path = OUT_DIR / 'prediction_binary.tif'

    processed = 0
    mining_pixels = 0
    valid_pixels = 0
    with rasterio.open(prob_path, 'w', **profile_prob) as dst_prob, \
         rasterio.open(binary_path, 'w', **profile_binary) as dst_binary:
        for ri in range(n_rows):
            for ci in range(n_cols):
                if LIMIT_PATCHES > 0 and processed >= LIMIT_PATCHES:
                    break
                window = Window(ci * PATCH_SIZE, ri * PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)
                patch = src.read(BAND_INDICES, window=window).astype(np.float32) * 10000.0
                tensor = torch.from_numpy(normalize(patch)).float().unsqueeze(0).to(device)
                with torch.no_grad():
                    out = task.model(tensor)
                    logits = out.output if hasattr(out, 'output') else out
                    mining_prob = torch.softmax(logits, dim=1)[0, 1].cpu().numpy().astype(np.float32)
                binary = (mining_prob >= MINING_THRESH).astype(np.uint8)
                dst_prob.write(mining_prob, 1, window=window)
                dst_binary.write(binary, 1, window=window)
                mining_pixels += int(binary.sum())
                valid_pixels += binary.size
                processed += 1
                if processed % 200 == 0 or processed == total:
                    print(f'  {processed}/{total}')
            if LIMIT_PATCHES > 0 and processed >= LIMIT_PATCHES:
                break

share = mining_pixels / max(valid_pixels, 1)
print(f'Done. Mining share={share:.4%}')
print(prob_path)
print(binary_path)
(OUT_DIR / 'inference_stats.txt').write_text(
    f'processed={processed}\nmining_share={share}\nckpt={CHECKPOINT_PATH}\nmosaic={MOSAIC_PATH}\n'
)



## Zelle 5: Zip für Download


In [ ]:
import shutil
zip_path = Path('/kaggle/working/inference_bono_full_ft')
# shutil.make_archive creates zip next to folder
shutil.make_archive('/kaggle/working/inference_bono_full_ft_bundle', 'zip', OUT_DIR)
print('Download: /kaggle/working/inference_bono_full_ft_bundle.zip')
!ls -lh /kaggle/working/inference_bono_full_ft_bundle.zip /kaggle/working/inference_bono_full_ft/
